In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import joblib

import warnings
warnings.filterwarnings("ignore")

from reiqa.ReIQA.networks.build_backbone import build_model 

c:\Users\tejas\anaconda3\envs\reiqa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [4]:
# class KonIQDataset(Dataset):
#     def __init__(self, csv_file, img_dir, transform=None):
#         self.data = pd.read_csv(csv_file)
#         self.img_dir = img_dir
#         self.transform = transform

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         img_name = os.path.join(self.img_dir, str(self.data.iloc[idx]['image_name']))
#         image = Image.open(img_name).convert('RGB')
#         mos = float(self.data.iloc[idx]['MOS'])

#         if self.transform:
#             image = self.transform(image)

#         return image, mos
    
class KonIQDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, str(self.data.iloc[idx]['image_name']))
        image = Image.open(img_name).convert('RGB')
        mos = float(self.data.iloc[idx]['MOS'])

        image_half = image.resize((image.size[0] // 2, image.size[1] // 2))

        if self.transform:
            img_tensor = self.transform(image)
            img_half_tensor = self.transform(image_half)

        return img_tensor, img_half_tensor, mos

In [5]:
class ReIQAOptions:
    def __init__(self):
        self.modal = 'RGB'
        self.jigsaw = False
        self.arch = 'resnet50'
        self.head = 'mlp'      # As specified in the README for evaluation
        self.feat_dim = 128
        self.mem = 'moco'

def load_reiqa_encoder(ckpt_path, device):
    """
    Initializes the Re-IQA backbone, loads the weights using DataParallel 
    (to match the author's saved state_dict keys), and returns the base encoder.
    """
    opt = ReIQAOptions()
    
    # Build the full architecture (Encoder + MLP Head)
    model, _ = build_model(opt)
    
    # The authors saved their checkpoints wrapped in DataParallel
    model = nn.DataParallel(model)
    
    # Load weights
    checkpoint = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(checkpoint['model'])
    
    # Extract ONLY the ResNet-50 encoder (drops the MLP head used for contrastive loss)
    # This aligns with `feat1 = model.module.encoder(...)` in their demo scripts
    encoder = model.module.encoder
    
    return encoder.to(device).eval()

In [6]:
@torch.no_grad()
def extract_features(content_model, quality_model, dataloader, device):
    all_features = []
    all_labels = []
    
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    for images, images_half, labels in tqdm(dataloader, desc="Extracting FP32 Multi-Scale Features"):
        images = images.to(device)
        images_half = images_half.to(device)
        
        c_norm = normalize(images)
        c_half_norm = normalize(images_half)
        
        # STRICT FP32 - No autocast here. It will be slower, but mathematically precise.
        feat_c = content_model(c_norm)
        feat_c_half = content_model(c_half_norm)
        
        feat_q = quality_model(images)
        feat_q_half = quality_model(images_half)
        
        feat_c = feat_c.view(feat_c.size(0), -1)
        feat_c_half = feat_c_half.view(feat_c_half.size(0), -1)
        feat_q = feat_q.view(feat_q.size(0), -1)
        feat_q_half = feat_q_half.view(feat_q_half.size(0), -1)
        
        feats_content = torch.cat((feat_c, feat_c_half), dim=1) 
        feats_quality = torch.cat((feat_q, feat_q_half), dim=1) 
        
        feats = torch.cat((feats_content, feats_quality), dim=1) 
            
        all_features.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())
        
    X = np.vstack(all_features)
    y = np.concatenate(all_labels)
    
    return X, y

# @torch.no_grad()
# def extract_features(content_model, quality_model, dataloader, device):
#     all_features = []
#     all_labels = []
    
#     normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
#     # Notice we unpack 3 items now: images, images_half, labels
#     for images, images_half, labels in tqdm(dataloader, desc="Extracting Multi-Scale Features"):
#         images = images.to(device)
#         images_half = images_half.to(device)
        
#         # Apply normalization ONLY for the content model
#         c_norm = normalize(images)
#         c_half_norm = normalize(images_half)
        
#         with torch.autocast(device_type='cuda', dtype=torch.float16):
#             # 1. Content Features (Original + Half)
#             feat_c = content_model(c_norm)
#             feat_c_half = content_model(c_half_norm)
            
#             # 2. Quality Features (Original + Half) -> NO NORMALIZATION
#             feat_q = quality_model(images)
#             feat_q_half = quality_model(images_half)
            
#             # Flatten to 1D
#             feat_c = feat_c.view(feat_c.size(0), -1)
#             feat_c_half = feat_c_half.view(feat_c_half.size(0), -1)
#             feat_q = feat_q.view(feat_q.size(0), -1)
#             feat_q_half = feat_q_half.view(feat_q_half.size(0), -1)
            
#             # Concatenate according to the authors' demo scripts: [orig, half]
#             feats_content = torch.cat((feat_c, feat_c_half), dim=1) # 4096
#             feats_quality = torch.cat((feat_q, feat_q_half), dim=1) # 4096
            
#             # Final 8192-dimensional vector per image
#             feats = torch.cat((feats_content, feats_quality), dim=1) 
            
#         all_features.append(feats.cpu().numpy())
#         all_labels.append(labels.numpy())
        
#     X = np.vstack(all_features)
#     y = np.concatenate(all_labels)
    
#     return X, y

In [7]:
def train_and_evaluate_regressor(X, y, num_splits=10):
    srocc_list = []
    plcc_list = []
    
    best_srocc_overall = -1.0
    final_best_model = None
    final_best_scaler = None

    # 1D Grid Search range: [10^-3, 10^3] as specified in Sec 4.4
    # alphas = np.logspace(-3, 3, 7) 
    alphas = np.logspace(-3, 3, 100)

    print(f"\n--- Starting {num_splits}-Split Evaluation (70/10/20 Protocol) ---")
    
    for i in range(num_splits):
        # Seed controls the random split for this iteration
        rng = np.random.RandomState(i)
        indices = np.arange(len(y))
        rng.shuffle(indices)
        
        # Calculate split indices for 70% Train, 10% Val, 20% Test
        train_end = int(0.70 * len(y))
        val_end = int(0.80 * len(y))
        
        train_idx = indices[:train_end]
        val_idx = indices[train_end:val_end]
        test_idx = indices[val_end:]
        
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        
        # Standardize Features (Fit ONLY on training data)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        
        best_alpha = None
        best_val_loss = float('inf')
        
        # 1D Grid Search on Validation Set
        for alpha in alphas:
            model = Ridge(alpha=alpha, random_state=i)
            model.fit(X_train_scaled, y_train)
            
            # Predict on validation set and compute L2 Loss (MSE)
            val_preds = model.predict(X_val_scaled)
            val_loss = mean_squared_error(y_val, val_preds)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_alpha = alpha
                
        # Retrain best model configuration on the training set
        best_model = Ridge(alpha=best_alpha, random_state=i)
        best_model.fit(X_train_scaled, y_train)
        
        # Evaluate on the 20% Test Set
        y_pred = best_model.predict(X_test_scaled)
        
        srocc, _ = spearmanr(y_pred, y_test)
        plcc, _ = pearsonr(y_pred, y_test)
        
        srocc_list.append(srocc)
        plcc_list.append(plcc)
        
        print(f"Split {i+1:02d}/{num_splits} | Best Alpha: {best_alpha:>7.3f} | Test SROCC: {srocc:.4f} | Test PLCC: {plcc:.4f}")
        
        # Save the absolute best performing model across all splits for your experiments
        if srocc > best_srocc_overall:
            best_srocc_overall = srocc
            final_best_model = best_model
            final_best_scaler = scaler
            
    print("\n=== Final Paper Metrics ===")
    print(f"Median SROCC: {np.median(srocc_list):.4f}")
    print(f"Median PLCC:  {np.median(plcc_list):.4f}")
    print("===========================\n")
    
    return final_best_model, final_best_scaler

In [9]:
csv_path = 'data/koniq10k/koniq10k_distributions_sets.csv'
img_dir = 'data/koniq10k/1024x768' 
save_path = 'reiqa_koniq_exact_pipeline_betterV2.pkl'

In [10]:
transform = transforms.Compose([
    transforms.ToTensor() 
])

In [11]:
dataset = KonIQDataset(csv_file=csv_path, img_dir=img_dir, transform=transform)

In [12]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)

In [13]:
content_ckpt = 'reiqa/ReIQA/savedCheckpoint/content_aware_r50.pth' # Update with your path
quality_ckpt = 'reiqa/ReIQA/savedCheckpoint/quality_aware_r50.pth' # Update with your path

content_model = load_reiqa_encoder(content_ckpt, DEVICE)
quality_model = load_reiqa_encoder(quality_ckpt, DEVICE)

In [14]:
X, y = extract_features(content_model, quality_model, dataloader, DEVICE)

Extracting FP32 Multi-Scale Features: 100%|██████████| 630/630 [26:47<00:00,  2.55s/it]


In [15]:
final_model, final_scaler = train_and_evaluate_regressor(X, y, num_splits=10)


--- Starting 10-Split Evaluation (70/10/20 Protocol) ---
Split 01/10 | Best Alpha: 1000.000 | Test SROCC: 0.8983 | Test PLCC: 0.9086
Split 02/10 | Best Alpha: 1000.000 | Test SROCC: 0.8987 | Test PLCC: 0.9082
Split 03/10 | Best Alpha: 1000.000 | Test SROCC: 0.8953 | Test PLCC: 0.9016
Split 04/10 | Best Alpha: 1000.000 | Test SROCC: 0.8848 | Test PLCC: 0.9078
Split 05/10 | Best Alpha: 1000.000 | Test SROCC: 0.9011 | Test PLCC: 0.9115
Split 06/10 | Best Alpha: 1000.000 | Test SROCC: 0.8980 | Test PLCC: 0.9076
Split 07/10 | Best Alpha: 1000.000 | Test SROCC: 0.8977 | Test PLCC: 0.9065
Split 08/10 | Best Alpha: 1000.000 | Test SROCC: 0.8942 | Test PLCC: 0.9047
Split 09/10 | Best Alpha: 1000.000 | Test SROCC: 0.8971 | Test PLCC: 0.9097
Split 10/10 | Best Alpha: 1000.000 | Test SROCC: 0.8918 | Test PLCC: 0.9091

=== Final Paper Metrics ===
Median SROCC: 0.8974
Median PLCC:  0.9080



In [16]:
joblib.dump({'scaler': final_scaler, 'model': final_model}, save_path)

['reiqa_koniq_exact_pipeline_betterV2.pkl']